# PCA of bipolar cell migration kinetics

Unsupervised PCA / clustering of single-cell migration tracks, with a
**statistically valid** comparison of the two cohorts: apically born (**aBC**)
and subapically born (**sBC**) bipolar cells.

- Clustering is done **blind** to identity (KMeans on standardised kinetic features).
- The significance test (**PERMANOVA**) is run on the **a priori cohorts**
  (read from the column-name prefix `A` / `S`), *not* on the clusters — this
  avoids circular inference.
- Agreement between the blind clusters and the cohorts is reported with the
  **Adjusted Rand Index** and a contingency table.
- **PERMDISP** checks that a PERMANOVA difference reflects a shift in *location*
  rather than a difference in *spread*.

**Input:** a CSV with a `Time` column (min) and one column per cell (position
in µm). Column names must start with `A` (apically born) or `S` (subapically born),
e.g. `A1`, `S12`. In the paper, tracks cover the first 100 min after division.

Five features are computed per cell: mean speed, net displacement,
Directionality Ratio, mean MSD and SD of instantaneous velocity.

Edit the **Config** cell, then run all cells. Outputs are written to `OUT_DIR`. Re-run the **Figure** cell to
regenerate the plot after changing colours or clustering.

Used for Fig. S4 in Cruz et al. (2026), *Development*.


## Imports

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu

## Config — edit here

In [ ]:
FILE_PATH     = "example_data/example_tracks.csv"   # Time column + one column per cell
SEP           = ";"        # column separator of the input CSV
DECIMAL       = ","        # decimal mark of the input CSV
OUT_DIR       = "results/pca"   # where figures and tables are saved
N_CLUSTERS    = 2          # KMeans clusters (blind exploration)
N_PERM        = 9999       # permutations for PERMANOVA / PERMDISP
RANDOM_STATE  = 42
COHORT_COLORS   = {"aBC": "#3A7FE0", "sBC": "#E05C3A"}
CLUSTER_MARKERS = ["o", "s", "^", "D", "v"]


## Functions

In [ ]:
def cohort_from_name(name):
    """Map a column name to its cohort from the first letter (A -> aBC, S -> sBC)."""
    c = name.strip().upper()
    if c.startswith("A"):
        return "aBC"
    if c.startswith("S"):
        return "sBC"
    return "unknown"


def compute_features(df):
    """Per-cell kinetic features from single-axis position tracks (NaN-safe)."""
    time = df.index.to_numpy(dtype=float)
    dt = np.diff(time)
    feats = {}
    for i, cell in enumerate(df.columns):
        pos = df.iloc[:, i].to_numpy(dtype=float)
        step = np.diff(pos)
        inst_vel = step / dt

        finite = np.isfinite(pos)
        net_disp = pos[finite][-1] - pos[finite][0] if finite.sum() >= 2 else np.nan
        total_path = np.nansum(np.abs(step))
        dir_ratio = abs(net_disp) / total_path if total_path > 0 else 0.0
        msd = [np.nanmean((pos[lag:] - pos[:-lag]) ** 2) for lag in range(1, len(pos))]

        feats[cell] = {
            "mean_speed":           np.nanmean(np.abs(inst_vel)),
            "net_displacement":     net_disp,
            "directionality_ratio": dir_ratio,
            "mean_MSD":             np.nanmean(msd),
            "velocity_std":         np.nanstd(inst_vel),
        }
    return pd.DataFrame(feats).T


def permanova(X, labels, n_perm=9999, random_state=42):
    """One-way PERMANOVA (Anderson 2001) on Euclidean distances.
    Tests whether group centroids differ; groups must be a priori.
    Returns (pseudo_F, p_value)."""
    rng = np.random.default_rng(random_state)
    D2 = squareform(pdist(X, metric="euclidean")) ** 2
    n = len(labels)
    groups = np.unique(labels)

    def pseudo_f(lbl):
        ss_total = D2.sum() / (2 * n)
        ss_within = 0.0
        for g in groups:
            idx = np.where(lbl == g)[0]
            if len(idx) >= 2:
                ss_within += D2[np.ix_(idx, idx)].sum() / (2 * len(idx))
        df_among, df_within = len(groups) - 1, n - len(groups)
        if df_within <= 0 or ss_within <= 0:
            return np.nan
        return ((ss_total - ss_within) / df_among) / (ss_within / df_within)

    obs = pseudo_f(labels)
    ge = sum(pseudo_f(rng.permutation(labels)) >= obs for _ in range(n_perm))
    return obs, (ge + 1) / (n_perm + 1)


def permdisp(X, labels, n_perm=9999, random_state=42):
    """Permutation test for homogeneity of multivariate dispersion (2 groups).
    Statistic: |difference in mean distance-to-centroid|. A non-significant
    result supports that a PERMANOVA difference is a location shift, not spread.
    Returns (mean_dispersion_by_group, statistic, p_value)."""
    rng = np.random.default_rng(random_state)
    groups = np.unique(labels)

    def disp_means(lbl):
        means = {}
        for g in groups:
            idx = np.where(lbl == g)[0]
            centroid = X[idx].mean(axis=0)
            means[g] = np.linalg.norm(X[idx] - centroid, axis=1).mean()
        return means

    obs_means = disp_means(labels)
    vals = list(obs_means.values())
    obs = abs(vals[0] - vals[1])
    ge = 0
    for _ in range(n_perm):
        m = list(disp_means(rng.permutation(labels)).values())
        if abs(m[0] - m[1]) >= obs:
            ge += 1
    return obs_means, obs, (ge + 1) / (n_perm + 1)


def holm_correction(pvals):
    """Holm-Bonferroni step-down adjusted p-values (family-wise error control)."""
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    n = len(p)
    adj = np.empty(n)
    running = 0.0
    for rank, idx in enumerate(order):
        running = max(running, (n - rank) * p[idx])
        adj[idx] = min(running, 1.0)
    return adj


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"

## Load data and assign cohorts

In [ ]:
df = pd.read_csv(FILE_PATH, sep=SEP, decimal=DECIMAL).set_index("Time")
cells = np.array(df.columns.tolist())
cohorts = np.array([cohort_from_name(c) for c in cells])

print(f"Loaded {df.shape[1]} cells x {df.shape[0]} timepoints")
for g in np.unique(cohorts):
    print(f"  {g}: {(cohorts == g).sum()} cells")

## Features, scaling and PCA

In [ ]:
feat_df = compute_features(df)
X = StandardScaler().fit_transform(feat_df.values)

pca = PCA()
X_pca = pca.fit_transform(X)
var = pca.explained_variance_ratio_ * 100
print(f"Explained variance: PC1 {var[0]:.1f}%, PC2 {var[1]:.1f}%, PC3 {var[2]:.1f}%")
feat_df.round(3)

## Blind clustering vs. known cohorts

KMeans is run **without** using the cohort labels. The Adjusted Rand Index and the contingency table describe how well the blind clusters recover the biological cohorts — this replaces any (circular) significance test on the clusters.

In [ ]:
km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20)
clusters = km.fit_predict(X)

ari = adjusted_rand_score(cohorts, clusters)
contingency = pd.crosstab(pd.Series(cohorts, name="cohort"),
                          pd.Series(clusters, name="cluster"))
print(contingency.to_string())
print(f"\nAdjusted Rand Index (cluster vs cohort): {ari:.3f}")

## Valid test: PERMANOVA on cohorts + dispersion check

PERMANOVA compares the **a priori cohorts** (aBC vs sBC). PERMDISP checks their spread is comparable, so a significant PERMANOVA reflects a shift in location.

In [ ]:
F, p = permanova(X, cohorts, n_perm=N_PERM, random_state=RANDOM_STATE)
print(f"PERMANOVA (aBC vs sBC): pseudo-F = {F:.3f}, p = {p:.4f} {stars(p)}  ({N_PERM} permutations)")
if p <= 1 / (N_PERM + 1):
    print(f"  (smallest p attainable with {N_PERM} permutations; report as p <= {1/(N_PERM+1):.4f})")

disp_means, disp_stat, disp_p = permdisp(X, cohorts, n_perm=N_PERM, random_state=RANDOM_STATE)
for g, m in disp_means.items():
    print(f"  mean distance to centroid, {g}: {m:.3f}")
print(f"PERMDISP: difference = {disp_stat:.3f}, p = {disp_p:.4f} {stars(disp_p)} "
      f"(ns supports equal spread)")

## Cohort separation along principal components (Holm-corrected)

In [ ]:
raw_p, Us = [], []
for pc in range(3):
    a = X_pca[cohorts == "aBC", pc]
    s = X_pca[cohorts == "sBC", pc]
    U, pmw = mannwhitneyu(a, s, alternative="two-sided")
    Us.append(U); raw_p.append(pmw)

adj_p = holm_correction(raw_p)
pc_stats = pd.DataFrame({
    "PC": ["PC1", "PC2", "PC3"],
    "variance_%": var[:3].round(1),
    "U": Us,
    "p_raw": raw_p,
    "p_holm": adj_p,
    "sig_holm": [stars(x) for x in adj_p],
})
print("Mann-Whitney U per PC, Holm-corrected across PC1-PC3")
print(pc_stats.to_string(index=False))

## Figure

Colour = cohort, marker = blind cluster. Re-run this cell after tweaking colours.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for g in np.unique(cohorts):
    for k in range(N_CLUSTERS):
        m = (cohorts == g) & (clusters == k)
        if m.any():
            ax.scatter(X_pca[m, 0], X_pca[m, 1],
                       c=COHORT_COLORS.get(g, "#888888"),
                       marker=CLUSTER_MARKERS[k % len(CLUSTER_MARKERS)],
                       s=70, alpha=0.85, edgecolors="k", linewidths=0.4,
                       label=f"{g} / cluster {k+1}")
ax.set_xlabel(f"PC1 ({var[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({var[1]:.1f}%)")
ax.set_title("Kinetic-feature PCA\ncolour = cohort, marker = blind cluster")
ax.axhline(0, color="#cccccc", lw=0.7); ax.axvline(0, color="#cccccc", lw=0.7)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=7, frameon=True)
fig.tight_layout()
plt.show()

## Save outputs

In [ ]:
out_dir = OUT_DIR
os.makedirs(out_dir, exist_ok=True)

fig.savefig(os.path.join(out_dir, "pca_cohort_cluster.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(out_dir, "pca_cohort_cluster.svg"), bbox_inches="tight")

feat_out = feat_df.copy()
feat_out["cohort"] = cohorts
feat_out["blind_cluster"] = clusters + 1
feat_out.to_csv(os.path.join(out_dir, "kinetic_features.csv"))

pd.DataFrame([
    {"test": "PERMANOVA (aBC vs sBC)", "statistic": F, "p_value": p},
    {"test": "PERMDISP (dispersion)", "statistic": disp_stat, "p_value": disp_p},
    {"test": "Adjusted Rand Index (cluster vs cohort)", "statistic": ari, "p_value": np.nan},
]).to_csv(os.path.join(out_dir, "statistics.csv"), index=False)

print(f"Saved figure, kinetic_features.csv, statistics.csv and pc_statistics.csv to: {out_dir}")
pc_stats.to_csv(os.path.join(out_dir, "pc_statistics.csv"), index=False)
